# 🧠 NeuroScan 3D Residual U-Net - Colab Inference Runner
This notebook sets up the GPU environment on Google Colab, pulls the latest repository code, executes sequential 3D volumetric inference on all cases in , and outputs segmentation masks to .

In [ ]:
# 1. Verify GPU availability
!nvidia-smi

In [ ]:
# 2. Clone or pull latest repository
import os
![ -d "SE374.R11-Group11" ] && (cd SE374.R11-Group11 && git pull) || git clone https://github.com/EmmYeuuKhoaaHocc/SE374.R11-Group11.git
%cd /content/SE374.R11-Group11/apps/worker/training
!pwd

In [ ]:
# 3. Install lightweight inference requirements
!pip install -q nibabel scipy scikit-learn tqdm matplotlib ipywidgets pandas

In [ ]:
# 4. Run Sequential Inference on all samples in data/ folder
!python inference.py

In [ ]:
# 5. Inspect generated segmentation volumes
!ls -lh predict/

In [ ]:
# 6. Quick inline visual check on generated masks
import glob, nibabel as nib, numpy as np, matplotlib.pyplot as plt

seg_files = sorted(glob.glob("predict/*_seg.nii.gz"))
print(f"Found {len(seg_files)} segmentation files: {seg_files}")

if seg_files:
    fig, axes = plt.subplots(1, len(seg_files), figsize=(5 * len(seg_files), 5))
    if len(seg_files) == 1: axes = [axes]
    
    for ax, p in zip(axes, seg_files):
        seg = nib.load(p).get_fdata().astype(np.uint8)
        mid = seg.shape[2] // 2
        # find axial slice with max tumor voxels
        slice_idx = int(np.argmax((seg > 0).sum(axis=(0, 1)))) if (seg > 0).any() else mid
        ax.imshow(np.rot90(seg[:, :, slice_idx]), cmap="tab10", vmin=0, vmax=9)
        cid = Path(p).name.replace("_seg.nii.gz", "")
        ax.set_title(f"{cid} (Z={slice_idx})")
        ax.axis("off")
    plt.tight_layout()
    plt.show()